In [1]:
import os
from pathlib import Path
import numpy as np
import pandas as pd

# ---------- Paths ----------
def find_base_dir():
    env = os.environ.get("PROJECT_DIR")
    if env and (Path(env) / "data" / "raw").exists():
        return Path(env)

    start = Path.cwd().resolve()
    for p in [start, *start.parents]:
        if (p / "data" / "raw").exists():
            return p

    try:
        import tkinter as tk
        from tkinter import filedialog
        root = tk.Tk()
        root.withdraw()
        root.attributes("-topmost", True)
        chosen = filedialog.askdirectory(title="Select the project folder (the one containing data/raw)")
        root.destroy()
    except Exception:
        chosen = input("Paste the project folder path (the one containing data/raw): ").strip().strip('"')

    if chosen and (Path(chosen) / "data" / "raw").exists():
        return Path(chosen)
    raise FileNotFoundError("data/raw not found in the selected folder.")

BASE_DIR = find_base_dir()
RAW_DIR = BASE_DIR / "data" / "raw"
OUT_DIR = BASE_DIR / "outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("BASE_DIR:", BASE_DIR)

def safe_save(save_fn, filename):
    """Try OUT_DIR first; if the OS rejects the write, fall back to the home folder."""
    try:
        save_fn(OUT_DIR / filename)
        print("Saved:", OUT_DIR / filename)
    except OSError as e:
        fallback = Path.home() / "part2b_outputs"
        fallback.mkdir(exist_ok=True)
        save_fn(fallback / filename)
        print("Could not write to", OUT_DIR, "-", e)
        print("Saved to", fallback / filename, "instead.")

# ---------- Data ----------
age_sex = pd.read_csv(RAW_DIR / "age_sex.csv")
scenarios = pd.read_csv(RAW_DIR / "projection_scenarios.csv")

proj_district = "D05"
sub = age_sex[age_sex["district_id"] == proj_district].copy()
ages = sorted(sub["age_start"].unique().tolist())

pop25 = {(r["age_start"], r["sex"]): float(r["population_2025"]) for _, r in sub.iterrows()}
surv = {(r["age_start"], r["sex"]): float(r["survival_5yr"]) for _, r in sub.iterrows()}
arr = {(r["age_start"], r["sex"]): float(r["arrivals_2025"]) for _, r in sub.iterrows()}
dep = {(r["age_start"], r["sex"]): float(r["departures_2025"]) for _, r in sub.iterrows()}
births_2025 = float(sub[sub["sex"] == "F"]["births_to_women_2025"].sum())

detail_rows = []
summary_rows = []

for _, sc in scenarios.iterrows():
    sc_name = sc["scenario"]
    fert_mult = float(sc["fertility_multiplier"])
    arr_mult = float(sc["arrivals_multiplier"])
    dep_mult = float(sc["departures_multiplier"])
    nb_surv = {"F": float(sc["newborn_survival_f"]), "M": float(sc["newborn_survival_m"])}

    births_5yr = 5.0 * births_2025 * fert_mult
    births_split = {"F": 0.49 * births_5yr, "M": 0.51 * births_5yr}

    pre_mig = {}
    nm_5yr_cell = {}
    pop2030_cell = {}

    for sex in ["F", "M"]:
        # Newborn survivors into ages 0-4
        pre_mig[(0, sex)] = births_split[sex] * nb_surv[sex]

        # Advance 0-4..80-84 into 5-9..80-84
        for idx in range(1, len(ages) - 1):
            a = ages[idx]
            prev_a = ages[idx - 1]
            pre_mig[(a, sex)] = pop25[(prev_a, sex)] * surv[(prev_a, sex)]

        # 85+ combines survivors from 80-84 and retention of existing 85+
        pre_mig[(85, sex)] = pop25[(80, sex)] * surv[(80, sex)] + pop25[(85, sex)] * surv[(85, sex)]

        # End-of-period migration adjustment across all age bands including 0-4
        for a in ages:
            nm_5yr = 5.0 * (arr[(a, sex)] * arr_mult - dep[(a, sex)] * dep_mult)
            nm_5yr_cell[(a, sex)] = nm_5yr
            pop2030_cell[(a, sex)] = pre_mig[(a, sex)] + nm_5yr

            detail_rows.append({
                "scenario": sc_name,
                "district_id": proj_district,
                "age_start": a,
                "sex": sex,
                "pop_2025": pop25[(a, sex)],
                "pre_mig_pop_2030": pre_mig[(a, sex)],
                "net_mig_5yr_end_adj": nm_5yr,
                "pop_2030": pop2030_cell[(a, sex)],
            })

    df_sc = pd.DataFrame([r for r in detail_rows if r["scenario"] == sc_name])
    tot_2030 = df_sc["pop_2030"].sum()
    tot_pre_mig = df_sc["pre_mig_pop_2030"].sum()
    tot_nm = df_sc["net_mig_5yr_end_adj"].sum()
    a0_4 = df_sc[df_sc["age_start"] == 0]["pop_2030"].sum()
    u15 = df_sc[df_sc["age_start"] < 15]["pop_2030"].sum()
    o65 = df_sc[df_sc["age_start"] >= 65]["pop_2030"].sum()

    # Accounting checks
    assert np.isclose(tot_pre_mig + tot_nm, tot_2030), f"Total mismatch in {sc_name}"
    assert (df_sc["pop_2030"] >= 0).all(), f"Negative cohort count in {sc_name}"

    summary_rows.append({
        "scenario": sc_name,
        "district_id": proj_district,
        "births_5yr": births_5yr,
        "pre_mig_total_2030": tot_pre_mig,
        "net_mig_5yr_total": tot_nm,
        "pop_2030_total": tot_2030,
        "pop_0_4_2030": a0_4,
        "share_under_15_pct": (u15 / tot_2030) * 100.0,
        "share_65_plus_pct": (o65 / tot_2030) * 100.0,
    })

detail_df = pd.DataFrame(detail_rows)
summary_df = pd.DataFrame(summary_rows)

# Show the summary table first, then save both files
display(summary_df)
safe_save(lambda p: detail_df.to_csv(p, index=False), "part2b_cohort_component_detail_d05.csv")
safe_save(lambda p: summary_df.to_csv(p, index=False), "part2b_cohort_component_summary_d05.csv")

BASE_DIR: C:\Users\SOOQ ELASER\Documents\GitHub\Group-4-Assignment-2-Two-


,scenario,district_id,births_5yr,pre_mig_total_2030,net_mig_5yr_total,pop_2030_total,pop_0_4_2030,share_under_15_pct,share_65_plus_pct
0,Low,D05,15263.6,160204.644099,6236.6,166441.244099,15127.200948,16.137949,13.746511
1,Base,D05,17345.0,162346.085551,10805.0,173151.085551,17442.642400,17.114559,13.519617
2,High,D05,19426.4,164548.428767,15373.4,179921.828767,19818.985616,18.046052,13.305147


Saved: C:\Users\SOOQ ELASER\Documents\GitHub\Group-4-Assignment-2-Two-\outputs\part2b_cohort_component_detail_d05.csv
Saved: C:\Users\SOOQ ELASER\Documents\GitHub\Group-4-Assignment-2-Two-\outputs\part2b_cohort_component_summary_d05.csv
